In [8]:
pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# 1. Simulate Locations (Latitude/Longitude)
# Let's pretend we have 20 delivery spots in Tokyo
def create_data_model():
    """Stores the data for the problem."""
    data = {}
    np.random.seed(42)
    data['locations'] = np.random.randint(0, 100, size=(20, 2)).tolist()
    data['time_windows'] = [(0,1000)]

    for _ in range(19):
        start = np.random.randint(0,800)
        end = start + 240
        data['time_windows'].append((start,end))
    data['service_time'] = 5
    data['depot'] = 0
    data['num_vehicles'] = 4
    return data

def compute_distance_matrix(locations):
    size = len(locations)
    matrix = {}
    for from_node in range(size):
        matrix[from_node] = {}
        for to_node in range(size):
            x1, y1 = locations[from_node]
            x2, y2 = locations[to_node]
            matrix[from_node][to_node] = abs(x1 - x2) + abs(y1 - y2)
    return matrix

data = create_data_model()
distance_matrix = compute_distance_matrix(data['locations'])

print(f"✅ Environment Ready.")
print(f"We have {len(data['locations'])} locations and {data['num_vehicles']} trucks.")
print(f"Distance from Warehouse (0) to Point 1: {distance_matrix[0][1]} km (approx)")

✅ Environment Ready.
We have 20 locations and 4 trucks.
Distance from Warehouse (0) to Point 1: 58 km (approx)


In [10]:
# args: (number of locations, number of vehicles, starting node)
manager = pywrapcp.RoutingIndexManager(
    len(data['locations']), 
    data['num_vehicles'], 
    data['depot']
)

routing = pywrapcp.RoutingModel(manager)

def time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
   
    return distance_matrix[from_node][to_node] + data['service_time']

time_callback_index = routing.RegisterTransitCallback(time_callback)

routing.SetArcCostEvaluatorOfAllVehicles(time_callback_index)

# --- THE 2024 PROBLEM LOGIC ---
# We add a "Dimension" to track distance.
# We set a HARD CAP (capacity) of 500 units per vehicle.
# If a route requires 501 units, the solver will NOT allow it.
dimension_name = 'Time'
routing.AddDimension(
    time_callback_index,
    1000,      # slack (max waiting time)
    2000,    # capacity (max work day)
    False,   # dont force start time to zero
    dimension_name
)
time_dimension = routing.GetDimensionOrDie(dimension_name)
for location_idx, (start, end) in enumerate(data['time_windows']):
    index = manager.NodeToIndex(location_idx)
    time_dimension.CumulVar(index).SetRange(start, end)
print('Time Dimension & Constraints Configured')

time_dimension.SetGlobalSpanCostCoefficient(100)
print("heijunka (leveling) logic applied")

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.time_limit.seconds = 30

print("🧩 Solving for optimized routes...")
solution = routing.SolveWithParameters(search_parameters)

def print_solution(data, manager, routing, solution):
    print(f'Objective: {solution.ObjectiveValue()} minutes')
    time_dimension = routing.GetDimensionOrDie('Time')
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        plan_output = f'Truck {vehicle_id} Schedule:\n'
        
        while not routing.IsEnd(index):
            time_var = time_dimension.CumulVar(index)
            arrival_time = solution.Min(time_var)
            node_index = manager.IndexToNode(index)
            window_start, window_end = data['time_windows'][node_index]
            plan_output += f'Loc {node_index} (Arrive: {arrival_time}m | Window: {window_start}-{window_end}) ->'
            index = solution.Value(routing.NextVar(index))
        
        time_var = time_dimension.CumulVar(index)
        arrival_time = solution.Min(time_var)
        plan_output += f'Depot (End: {arrival_time}m)\n'
        print(plan_output)
        print('-' * 50)
if solution:
    print_solution(data, manager, routing, solution)
else:
    print("❌ No solution found! (Even with 30s timeout). constraints are likely mutually exclusive.")

Time Dimension & Constraints Configured
heijunka (leveling) logic applied
🧩 Solving for optimized routes...
Objective: 63314 minutes
Truck 0 Schedule:
Loc 0 (Arrive: 167m | Window: 0-1000) ->Loc 14 (Arrive: 253m | Window: 13-253) ->Loc 15 (Arrive: 270m | Window: 241-481) ->Loc 19 (Arrive: 339m | Window: 339-579) ->Loc 17 (Arrive: 379m | Window: 345-585) ->Depot (End: 405m)

--------------------------------------------------
Truck 1 Schedule:
Loc 0 (Arrive: 167m | Window: 0-1000) ->Loc 5 (Arrive: 215m | Window: 130-370) ->Loc 3 (Arrive: 243m | Window: 243-483) ->Loc 4 (Arrive: 504m | Window: 504-744) ->Loc 16 (Arrive: 776m | Window: 776-1016) ->Depot (End: 792m)

--------------------------------------------------
Truck 2 Schedule:
Loc 0 (Arrive: 167m | Window: 0-1000) ->Loc 8 (Arrive: 227m | Window: 20-260) ->Loc 10 (Arrive: 273m | Window: 273-513) ->Loc 9 (Arrive: 332m | Window: 166-406) ->Loc 13 (Arrive: 381m | Window: 315-555) ->Loc 2 (Arrive: 566m | Window: 566-806) ->Loc 11 (Arrive